In [ ]:
#Installs
!pip install xlrd
!pip install openpyxl

In [ ]:
#Imports
import numpy as np
import pandas as pd 
import pycountry
import csv

import random
import string

from datetime import datetime
from contextlib import redirect_stdout #For the log files

In [ ]:
#file_name = "../input/promed/Ventas Regulares Corregido 2.xlsx"
#df = pd.read_excel(file_name, engine='openpyxl', sheet_name='9ba16c51a1fe738624fe8dd92edc9b4')
file_name = "../input/promed/MARCAS-PARCIALRECODED-revisionPromed.xlsx"
df = pd.read_excel(file_name, sheet_name='PromedDepuradas')

In [ ]:
df.head(26)

In [ ]:
df.iloc[656,1] = 'MXXXX-XXXX'
df['ReConCat'][14:19] = 'XXXX'

df.drop([df.index[630], df.index[631],], inplace=True)
#df = df.reset_index( drop=True)

In [ ]:
def checkDuplicate(lists):
    duplicate={}
    for i in lists:
            ## checking whether the item is already present in dictionary or not
            ## increasing count if present
            ## initializing count to 1 if not present
        duplicate[i] = duplicate.get(i,0) + 1

    return [k for k, v in duplicate.items() if v > 1]

In [ ]:
#vl_0 = [i.split('-')[0] for i in df['KBOXcode'][19:] if isinstance(i, str)]
##vl_1 = [i.split('-')[1] for i in df['KBOXcode'][19:] if isinstance(i, str)]
#
#vl_0_m = [i.split('M')[1] for i in vl_0 if isinstance(i, str)]
##vl_0_m_mod = [ 'M'+ i.zfill(5) for i in vl_0_m]
#vl_0_m_mod = []
#
#for i in vl_0_m:
#    if i == 'XXXX':
#        num = 'M'+ i
#        vl_0_m_mod.append(i)
#    else:
#        num = 'M'+ i.zfill(4)
#        vl_0_m_mod.append(num)
        

#vl_fin = [(x +'-' + y) for x in vl_0_m_mod for y in vl_1]

In [ ]:
##1 - Num Secuencial Comercial
#def marcaNumReencoder(table):
#
#    try:
#        # Assigning numerical values and storing in another column
#        num_comercial = table['ReConCat']
#        codesStr = 'M' + num_comercial.astype('string').str.zfill(5)
#        print(datetime.now().strftime("%d-%m-%Y %H:%M:%S"), 'Finished marca num encoding')
#        return codesStr
#    
#    except:
#        print(datetime.now().strftime("%d-%m-%Y %H:%M:%S"), 'Error during encoding')
#        pass
#        

In [ ]:
#1 - Num Secuencial Comercial
def marcaNumReencoder(table):

    try:
        # Assigning numerical values and storing in another column
        num_comercial =  table['DESCRIPCION_MARCA'].astype('category')

        codes = num_comercial.cat.codes
        cats = num_comercial.cat.categories
        codesStr = 'M' + codes.astype('string').str.zfill(4)
        
        print(datetime.now().strftime("%d-%m-%Y %H:%M:%S"), 'Finished marca num encoding')
        return codesStr
    
    except:
        print(datetime.now().strftime("%d-%m-%Y %H:%M:%S"), 'Error during encoding')
        pass
        

In [ ]:
a = marcaNumReencoder(df)

In [ ]:
#2 - Alias Marca
def random_chars(alias_len):
       return ''.join(random.choice(string.ascii_uppercase) for letter in range(alias_len))

def marcaAliaspReencoder(table):

    try:
        #Save the marca and alias equivalences
        marca_dict = {}
        alias_list = []
        
        #ky = [i for i in table['DESCRIPCION_MARCA'][19:] if isinstance(i, str)]
        #vl = [i.split('-')[1] for i in table['KBOXcode'][19:] if isinstance(i, str)]
        #marca_dict_extra = dict(zip(ky, vl))

        for marca in table['DESCRIPCION_MARCA']:
            alias = random_chars(4)

            if (alias not in marca_dict.values()): #& (alias not in marca_dict_extra):
                marca_dict[marca] = alias
                alias_list.append(alias)

            else:
                while(alias in marca_dict.values()):
                    alias = random_chars(4)

                marca_dict[marca] = alias
                alias_list.append(alias)

        return (alias_list, marca_dict) 
                
        print(datetime.now().strftime("%d-%m-%Y %H:%M:%S"), 'Finished marca alias encoding')
        return espStr
    
    except:
        print(datetime.now().strftime("%d-%m-%Y %H:%M:%S"), 'Error during encoding')
        pass
        

In [ ]:
b = marcaAliaspReencoder(df)

In [ ]:
def marcaeEncoder(table):
    try:
        with open('logfile.txt', 'a') as f:
            with redirect_stdout(f):                
                
                table['Codigo_KBOX_Marca'] = marcaNumReencoder(table) + '-' + marcaAliaspReencoder(table)[0]
                print(datetime.now().strftime("%d-%m-%Y %H:%M:%S"), 'Finished article complete sequence encoding. Undefined')
                return table['Codigo_KBOX_Marca']
                    
    except: 
        with open('logfile.txt', 'a') as f:
            with redirect_stdout(f):
                print(datetime.now().strftime("%d-%m-%Y %H:%M:%S"), 'Error during encoding')
        pass  

In [ ]:
marcaeEncoder(df)

In [ ]:
df.head(20)

In [ ]:
marcaeEncoder(df)
df.to_excel("recoded_marcas.xlsx", sheet_name='Recoded', index = False)